In [14]:
import requests
import time
from bs4 import BeautifulSoup
import re

In [ ]:
def fetch_abstract(index, _max_length):
    url = 'https://www.tv-asahi.co.jp/doraemon/story/{0}/'.format(index)
    html_text = requests.get(url).text
    soup = BeautifulSoup(html_text, 'html.parser')

    # 整構造でない場合はテキスト全部取得する
    p_list = soup.find_all('p', class_="txt_idt") if len(soup.find_all('p', class_="txt_idt")) > 0 else soup.find('div', class_='contentsarea').find_all('p')
    abstracts_list = [
        asbstract_dom.text
        for asbstract_dom in p_list
    ]
    
    dateDom = soup.find('p', class_='date')
    broadcasting_date = re.sub(r'[\[\]放送]', '', dateDom.text)

    t_list = soup.find_all('h2', class_='story-title') if (len(soup.find_all('h2', class_='story-title')) > 0) else soup.find('p').find('strong').text
    title_list = [
        title_dom.text
        for title_dom in t_list
    ]

    return {
        "index": index,
        "title": title_list,
        "broadcasting_date": broadcasting_date,
        # 50文字ある文章のみ抽出 [asbstract for asbstract in abstracts_list if len(asbstract) > max_length
        "abstracts_list": abstracts_list
    }


In [ ]:
start = 325
end = 857
max_length = 50

stories_list = []
for i in range(start, end+1):
    s_zero = str(i).zfill(4)
    print(i)
    stories_list.append(
        fetch_abstract(s_zero, max_length)
    )
    time.sleep(5)

325
326


In [17]:
flattened_stories = []

for story in stories_list:
    for i, (title, abstract) in enumerate(zip(story["title"], story["abstracts_list"])):
        flattened_stories.append({
            "index": "{0}_{1}".format(story["index"], i),
            "broadcasting_date": story["broadcasting_date"],
            "title": title,
            "abstract": abstract
        })

In [18]:
import json
with open("output_storiesv3.json", 'w') as f:
    json.dump(flattened_stories, f)